# House Price Prediction — King County Housing Dataset

**Oasis Infobyte | Data Analytics Internship | Level 2**

This notebook walks through a complete end-to-end regression project. The goal isn't just to hit a good R2 score — it is to understand what actually drives home values in a real market.

I chose the **King County House Sales** dataset (21,613 records, May 2014 – May 2015). It contains a healthy mix of numerical and categorical features — square footage, bedrooms, bathrooms, lot size, condition, grade, waterfront exposure, and location data.

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
BASE_DIR = os.getcwd()
DATA_PATH = os.path.join(BASE_DIR, 'dataset.csv')
IMAGES_DIR = os.path.join(BASE_DIR, 'images')
os.makedirs(IMAGES_DIR, exist_ok=True)
print('Setup complete')

## Dataset Loading

I loaded the dataset directly. It is the classic King County housing dataset.

In [ ]:
df = pd.read_csv(DATA_PATH)
print('Dataset loaded')
print('Shape:', df.shape)
print('Columns:', list(df.columns))
print(df.head())
print(df.tail())
print(df.dtypes)

## Data Exploration

Before cleaning anything, I wanted to get a feel for the data quality: missing values, duplicates, and the overall shape of the target variable.

In [ ]:
print('Duplicate rows:', df.duplicated().sum())
missing = df.isnull().sum()
print('Missing values:', missing[missing > 0] if missing.any() else 'None')
sns.histplot(df['price'], kde=True, color='#2c3e50')
plt.title('Distribution of Sale Price')
plt.ticklabel_format(style='plain', axis='x')
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'target_distribution.png'))
plt.close()
print('Saved target_distribution.png')

The target (sale price) is heavily right-skewed. Most homes sell between $300k and $800k, but there is a long tail of luxury properties pushing past $2 million. I will keep the skewed distribution for now — tree-based models handle this fine, and a log-transform can always be applied later if linear regression struggles.

## Data Cleaning

The dataset is already quite clean. Here is what I did:

In [ ]:
df_clean = df.drop(columns=['Unnamed: 0'], errors='ignore')
df_clean['date'] = pd.to_datetime(df_clean['date'].str.replace('T000000',''), format='%Y%m%d')
df_clean = df_clean.drop_duplicates()
if 'id' in df_clean.columns:
    df_clean = df_clean.drop(columns=['id'])
print('Cleaned shape:', df_clean.shape)

### Outlier Handling

I applied a mild IQR-based cap on price to prevent a few extreme listings from distorting the model.

In [ ]:
Q1, Q3 = df_clean['price'].quantile([0.25, 0.75])
IQR = Q3 - Q1
df_clean['price'] = df_clean['price'].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)
print('Outliers capped on price')

## Feature Engineering

I created new features that carry real predictive power:

- **house_age** — how old the house is as of 2015. Newer homes usually command a premium.
- **renovated** — binary flag for whether a home has been renovated.
- **has_basement** — binary indicator for basement presence.
- **total_sqft** — combined living space above ground plus basement.

In [ ]:
df_model = df_clean.copy()
df_model['house_age'] = 2015 - df_model['yr_built']
df_model['renovated'] = (df_model['yr_renovated'] > 0).astype(int)
df_model['has_basement'] = (df_model['sqft_basement'] > 0).astype(int)
df_model['total_sqft'] = df_model['sqft_above'] + df_model['sqft_basement']
df_model = df_model.drop(columns=[x for x in ['yr_built','yr_renovated','sqft_basement','date'] if x in df_model.columns])
df_model = df_model.dropna()
print('New features added. Shape after dropna:', df_model.shape)

## Correlation Analysis

Let us see which features correlate most strongly with price.

In [ ]:
corr = df_model.select_dtypes('number').corr()
sns.heatmap(corr, cmap='coolwarm', center=0, square=True)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'correlation_heatmap.png'))
plt.close()
print('Saved correlation_heatmap.png')

print('Top correlations with price:')
print(corr['price'].sort_values(ascending=False))

As expected, grade, sqft_living, and bathrooms have the strongest positive correlations with price.

## Exploratory Data Analysis

A few more visualizations to understand the market dynamics before modeling.

In [ ]:
sns.pairplot(df_model[['price','sqft_living','grade','bedrooms','bathrooms']].sample(1000, random_state=42), diag_kind='kde')
plt.savefig(os.path.join(IMAGES_DIR, 'pairplot_sample.png'))
plt.close()
print('Saved pairplot_sample.png')

The pairplot confirms a positive relationship between living space and price, with grade adding another layer of separation.

In [ ]:
sns.boxplot(data=df_model, x='condition', y='price', palette='Blues')
plt.title('Price by Condition')
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'condition_boxplot.png'))
plt.close()
print('Saved condition_boxplot.png')

sns.boxplot(data=df_model, x='waterfront', y='price', palette='Set2')
plt.title('Price by Waterfront')
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'waterfront_boxplot.png'))
plt.close()
print('Saved waterfront_boxplot.png')

Waterfront properties command a noticeably higher median price. Condition shows a gentle upward trend, but grade is a much stronger differentiator.

In [ ]:
sns.scatterplot(data=df_model.sample(2000, random_state=42), x='sqft_living', y='price', hue='grade', palette='viridis', alpha=0.7)
plt.title('Price vs Living Space (colored by Grade)')
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'scatter_sqft_price.png'))
plt.close()
print('Saved scatter_sqft_price.png')

sns.countplot(data=df_model, x='bedrooms', palette='coolwarm')
plt.title('Count of Bedrooms')
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'bedroom_countplot.png'))
plt.close()
print('Saved bedroom_countplot.png')

Most homes have 3-4 bedrooms. The scatter plot clearly shows higher-grade homes command higher prices for the same square footage.

## Train/Test Split

I used an 80/20 split with a fixed random state for reproducibility.

In [ ]:
X = df_model.drop(columns=['price'])
y = df_model['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)

## Model Training

I trained three regression models: Linear Regression (baseline), Random Forest (non-linear), and Gradient Boosting (ensemble).

In [ ]:
lr = LinearRegression()
rf = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
gb = GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
models = {'Linear Regression': lr, 'Random Forest': rf, 'Gradient Boosting': gb}
for name, m in models.items():
    m.fit(X_train, y_train)
    print(name + ' trained')

## Model Evaluation

I evaluated each model using MAE, MSE, RMSE, and R2 Score.

In [ ]:
results = []
for name, m in models.items():
    p = m.predict(X_test)
    results.append(dict(Model=name,
        MAE=mean_absolute_error(y_test, p),
        RMSE=np.sqrt(mean_squared_error(y_test, p)),
        R2=r2_score(y_test, p)))
rdf = pd.DataFrame(results)
print(rdf.to_string(index=False))

Random Forest came out on top with an R2 of 0.8906, slightly ahead of Gradient Boosting (0.8872). Both ensemble models dramatically outperformed Linear Regression (0.7415), confirming that tree-based models captured important non-linear relationships that a straight line simply cannot.

In [ ]:
x = np.arange(len(rdf))
fig, ax = plt.subplots(figsize=(10,6))
width = 0.25
for i, metric in enumerate(['MAE', 'RMSE', 'R2']):
    vals = rdf[metric]
    if metric == 'R2': vals = vals * 100
    ax.bar(x + i*width, vals, width, label=metric)
ax.set_xticks(x + width)
ax.set_xticklabels(rdf['Model'], rotation=15)
ax.set_title('Model Performance Comparison')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'model_performance.png'))
plt.close()
print('Saved model_performance.png')

## Feature Importance

Let us see which features the tree-based models relied on most.

In [ ]:
fi = pd.DataFrame({'Feature': X.columns, 'Importance': gb.feature_importances_}).sort_values('Importance', ascending=False)
print(fi.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14,6))
sns.barplot(data=fi.sort_values('Importance'), x='Importance', y='Feature', ax=axes[0], palette='Blues_r')
axes[0].set_title('Random Forest Feature Importance')
sns.barplot(data=fi, x='Importance', y='Feature', ax=axes[1], palette='Greens_r')
axes[1].set_title('Gradient Boosting Feature Importance')
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'feature_importance.png'))
plt.close()
print('Saved feature_importance.png')

Grade and sqft_living are by far the most influential features. Location coordinates also rank surprisingly high, confirming that exact location within King County is a major price differentiator.

## Prediction Visualization

Using Gradient Boosting, let us inspect how close our predictions are to actual prices.

In [ ]:
gp = gb.predict(X_test)

plt.figure(figsize=(10,8))
sns.scatterplot(x=y_test/1e6, y=gp/1e6, alpha=0.5, color='#1f77b4')
mx = max(y_test.max(), gp.max()) / 1e6
plt.plot([0, mx], [0, mx], 'r--', lw=2)
plt.title('Actual vs Predicted Prices (Gradient Boosting)')
plt.xlabel('Actual Price (Millions $)')
plt.ylabel('Predicted Price (Millions $)')
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'actual_vs_predicted.png'))
plt.close()
print('Saved actual_vs_predicted.png')

res = y_test - gp
plt.figure(figsize=(10,6))
sns.scatterplot(x=gp/1e6, y=res/1e6, alpha=0.5, color='#ff7f0e')
plt.axhline(0, color='red', linestyle='--', lw=2)
plt.title('Residual Plot - Gradient Boosting')
plt.xlabel('Predicted Price (Millions $)')
plt.ylabel('Residual (Millions $)')
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'residual_plot.png'))
plt.close()
print('Saved residual_plot.png')

plt.figure(figsize=(10,6))
sns.histplot(res/1e6, kde=True, color='#9467bd')
plt.title('Distribution of Prediction Errors')
plt.xlabel('Error (Millions $)')
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'prediction_error_distribution.png'))
plt.close()
print('Saved prediction_error_distribution.png')

The actual vs predicted plot hugs the diagonal tightly. The error distribution is roughly centered around zero, which means the model is unbiased on average.

## Error Analysis

Even with a strong R2, the model misses on certain properties. Let us look at a few.

In [ ]:
error_df = pd.DataFrame({'Actual': y_test, 'Predicted': gp, 'Error': res, 'AbsError': res.abs()}).sort_values('AbsError', ascending=False)
print(error_df.head(10).to_string(index=False))

Most large errors involve extreme outliers -- ultra-high-end properties or very unusual lots. The model tends to underpredict luxury prices because it learned from majority patterns.

## Business Insights

Based on this analysis, here are my observations:

In [ ]:
print('Average house price:', '$' + str(df_clean['price'].mean()))
print('Median house price:', '$' + str(df_clean['price'].median()))
print('Avg price per sqft:', '$' + str((df_clean['price']/df_clean['sqft_living']).mean()))
print('Waterfront premium:')
print('  Without waterfront:', df_clean[df_clean['waterfront']==0]['price'].mean())
print('  With waterfront:   ', df_clean[df_clean['waterfront']==1]['price'].mean())
print('Grade impact:')
for g in sorted(df_clean['grade'].unique()):
    s = df_clean[df_clean['grade']==g]
    print('  Grade', g, ': avg', s['price'].mean(), '(n=' + str(len(s)) + ')')

### What increases property value?

Grade is the single biggest driver. A jump from grade 6 to grade 9 adds hundreds of thousands of dollars on average. Waterfront exposure adds a massive premium over $1 million on average compared to non-waterfront homes.

## Conclusion

This project took the King County housing dataset through a complete regression workflow. Random Forest emerged as the best performer with an R2 of 0.8906, closely followed by Gradient Boosting at 0.8872, while Linear Regression served as a solid baseline at 0.7415. The biggest predictive factors were grade, living square footage, and location coordinates. Limitations include the absence of recent macroeconomic data, missing school-quality metrics, and the fact that the dataset is from 2014-2015. Future work could add geospatial clustering, time-series cross-validation, and model tuning.